# CH12 &mdash; the same filter in softwareCH11 ended with a table: the accelerator did Sobel on a 1080p frame in 11.4 msand OpenCV on four A53s took 72 ms, so the PL was 6.3&times; faster. Bothnumbers were *per frame*, because both were doing the same thing &mdash; take aframe, filter it, give it back.This chapter's filter is not doing that. It is in the pixel path, so its cost isnot 6&times; smaller than software: it is **zero**. The frame arrives in DDRalready filtered, at whatever rate the camera sends it, and turning the filter ondoes not make the next frame arrive any later.That is what this notebook measures. It is worth being clear about what is beingcompared, because it is not a like-for-like race:| | what the A53s do per frame | what the PL does ||---|---|---|| PL filter | copy the frame to the display | filter, in the pixel path || NumPy filter | luma, Sobel, saturate, copy | forward the stream untouched || OpenCV filter | the same, with NEON | forward the stream untouched |**Prerequisites**: the same as the first notebook, and run it first if thecamera is not already up.

In [ ]:
import sys, timeimport numpy as npfrom pynq import Overlayfrom pynq.lib.video import VideoModesys.path.insert(0, '.')import sobel_refol    = Overlay('sobel_stream.bit')mipi  = ol.mipisobel = ol.mipi.sobelW, H = 1280, 720sobel.register_map.img_width  = Wsobel.register_map.img_height = Hsobel.register_map.mode       = sobel_ref.MODE_COLORmipi.configure(VideoMode(W, H, 24))mipi.start()frame = mipi.readframe().copy()print(f'{W}x{H}, {frame.nbytes/1e6:.2f} MB a frame')

## Filtering one frame, three waysThe NumPy version is the bit-exact one &mdash; the same integer arithmetic asthe RTL, which is what makes it usable as a golden model. OpenCV is faster anddoes not match: `cvtColor` uses different luma coefficients and differentrounding, and `Sobel` extends the frame edge instead of blacking it out. Thepure-Python version is here to be timed on a fraction of the frame andextrapolated, because timing the whole thing would take most of a minute.

In [ ]:
def bench(fn, *args, repeat=5):    best = float('inf')    for _ in range(repeat):        t0 = time.perf_counter()        fn(*args)        best = min(best, time.perf_counter() - t0)    return bestt_numpy = bench(sobel_ref.filter_frame, frame, sobel_ref.MODE_SOBEL)print(f'NumPy (bit-exact)   {t_numpy*1e3:8.2f} ms')try:    t_cv = bench(sobel_ref.filter_frame_opencv, frame, sobel_ref.MODE_SOBEL)    print(f'OpenCV (approx)     {t_cv*1e3:8.2f} ms')except ImportError:    t_cv = None    print('OpenCV not installed')rows = 16strip = frame[:rows].copy()t_strip = bench(sobel_ref.filter_frame_naive, strip, sobel_ref.MODE_SOBEL, repeat=1)t_py = t_strip * H / rowsprint(f'pure Python         {t_py*1e3:8.0f} ms  (extrapolated from {rows} rows)')

## What the PL costsNothing, and the way to show that is to measure the frame rate with the filteron and with it off. If filtering cost anything the numbers would differ.

In [ ]:
def measure_fps(mode, frames=120):    sobel.register_map.mode = mode    for _ in range(3):        mipi.readframe()    t0 = time.perf_counter()    for _ in range(frames):        mipi.readframe()    return frames / (time.perf_counter() - t0)for mode, name in sobel_ref.MODE_NAMES.items():    print(f'{name:<10} {measure_fps(mode):5.1f} fps')

The four numbers should be the same to within measurement noise. They are thecamera's frame rate, not the filter's throughput: the filter is upstream of theVDMA and never in the way.For the record of what it *could* sustain: two pixels per clock at 300 MHz is600 Mpixel/s, which is 650 frames a second at 720p. The camera sends 60, and the2-lane RAW10 link could not carry more than about 134 Mpixel/s even if it did.The filter is running at roughly a tenth of its capacity, and the first thingthat would stop it going faster is the line buffers -- sized for 1920 pixels --rather than the arithmetic.

## Frame rate when the A53s do the filteringNow the same loop with the PL in passthrough and the filter in software. This isthe comparison that matters, because it is what the whole design is for.

In [ ]:
def measure_sw_fps(fn, frames=30):    sobel.register_map.mode = sobel_ref.MODE_COLOR    for _ in range(3):        mipi.readframe()    t0 = time.perf_counter()    for _ in range(frames):        f = mipi.readframe()        fn(f, sobel_ref.MODE_SOBEL)    return frames / (time.perf_counter() - t0)fps_pl = measure_fps(sobel_ref.MODE_SOBEL)fps_np = measure_sw_fps(sobel_ref.filter_frame)print(f'PL streaming filter   {fps_pl:6.1f} fps')print(f'NumPy on the A53s     {fps_np:6.1f} fps')try:    fps_cv = measure_sw_fps(sobel_ref.filter_frame_opencv)    print(f'OpenCV on the A53s    {fps_cv:6.1f} fps')except ImportError:    fps_cv = None

## The summary tableFill this in with what your board actually printed rather than trusting thenumbers in the chapter: the frame rate depends on the light (the OV5640 lengthensits exposure in a dim room and sends fewer frames), and the software numbersdepend on what else the A53s are doing.

In [ ]:
print(f'{"":<22}{"ms / frame":>12}{"fps":>8}')print(f'{"PL streaming filter":<22}{0.0:>12.2f}{fps_pl:>8.1f}')print(f'{"NumPy (bit-exact)":<22}{t_numpy*1e3:>12.2f}{fps_np:>8.1f}')if t_cv is not None:    print(f'{"OpenCV (approximate)":<22}{t_cv*1e3:>12.2f}{fps_cv:>8.1f}')print(f'{"pure Python":<22}{t_py*1e3:>12.0f}{1/t_py:>8.2f}')

## A fairer comparison, if you want oneEverything above races the PL against software on *different* frames &mdash; thecamera keeps sending, and neither path sees quite what the other saw. If thatbothers you, switch the filter's input to DDR and feed both paths the sameframes:```pythonimport video_source as vsplayer = vs.FramePlayer(mipi, W, H).start()vs.camera_enabled(mipi, False)pattern = vs.test_pattern(W, H)sobel.register_map.mode = sobel_ref.MODE_SOBELplayer.play(pattern); pl_out = mipi.readframe()      # filtered in the PLsw_out = sobel_ref.filter_frame(pattern, sobel_ref.MODE_SOBEL)print(sobel_ref.compare(pl_out, sw_out))             # expect (0, 0)```That is a correctness check rather than a speed one &mdash; and it is the checkthe first notebook makes properly. The timing story does not change: the PLfilter still costs nothing, because it is still not in the frame loop.

## What CH11 and CH12 are each good forCH11's memory-mapped accelerator is the right shape when the data is already inDDR and you want it filtered once: an image you loaded, a buffer anotheraccelerator produced, a frame you are about to compress. It is a function call.Its cost scales with how often you call it.CH12's streaming filter is the right shape when the data is *moving*. It costsone line of latency and no frames per second, and it holds that up to about tentimes the pixel rate this camera can produce. What it cannot do is filtersomething already sitting in memory &mdash; there is no port to feed it from.Neither is a better accelerator. They are the same arithmetic wearing differentinterfaces, and the interface is the whole design decision.

In [ ]:
mipi.stop()print('camera released')